# 04 - Centrality Analysis

This notebook ranks the stations of the Israeli public-transport network by several
centrality measures and asks whether the different measures agree with each other.
We compute degree and weighted degree (service volume), weighted PageRank on the
directed trip graph (flow importance), approximate betweenness on the largest
connected component (transfer / bridging importance) and harmonic centrality
(reachability). We then compare the rankings with a Spearman correlation matrix and,
critically, we measure how noisy the sampled betweenness estimate actually is before
anyone builds a "critical station" threshold on top of it.

**Research question.** Do cheap local measures (degree, weighted degree, PageRank)
identify the same stations as the expensive global measure (betweenness)? If they do,
the cheap measures are a good proxy; if they do not, the network has structurally
important stations that carry little traffic.

**Inputs**
- `outputs/nb/02_graph_construction/` - the node table and the directed edge table
  produced by notebook `02_graph_construction` (station attributes; `from_stop`,
  `to_stop`, trip frequency per segment).
- No raw GTFS file is required. `stop_times.txt` (816 MB) is only touched by an
  optional sanity check at the very end, and only if it already happens to be on disk.

**Outputs** (all under `outputs/nb/04_centrality_analysis/`)
- `tables/stop_metrics.csv` - one row per station with every centrality measure.
- `tables/top_degree.csv`, `top_weighted_degree.csv`, `top_pagerank.csv`,
  `top_approx_betweenness.csv`, `top_approx_harmonic.csv` - top-N rankings.
- `tables/centrality_correlation_spearman.csv` - Spearman matrix between measures.
- `tables/betweenness_stability.csv` - how much the betweenness ranking moves when
  only the random sample of sources changes.
- `figures/` - top-N bar charts per measure, correlation heatmap, degree-vs-measure
  scatter plots, a map of the top betweenness stations, and the seed-vs-seed
  stability scatter.

**Honesty note up front.** Betweenness here is a *k*-sample approximation
(default `K_BETWEENNESS = 300` sampled sources over roughly 30,000 nodes, i.e. about
1% of the possible sources). It is noisy, especially in the tail, and any downstream
"critical station" rule such as a p90 threshold on betweenness inherits that noise.
We quantify this rather than hide it.

## 1. Environment bootstrap

The next cell makes the notebook runnable both locally and on Google Colab. It finds
the repository root by walking up from the current directory looking for the GTFS data
folder, and if it cannot find it (i.e. we are on a fresh Colab machine) it clones the
repository. It also installs missing packages only - nothing is reinstalled if it is
already available. Every later cell assumes `REPO`, `DATA` and `OUT` exist.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Imports, output folders and tunable constants

All the parameters that control runtime live in this one cell so a grader can lower
them without reading the rest of the notebook.

- `K_BETWEENNESS = 300` - number of randomly sampled source nodes for the approximate
  betweenness. Exact betweenness on this graph needs a BFS from every one of ~30,000
  nodes (hours). With `k = 300` a run takes roughly 1-3 minutes.
- `RUN_STABILITY_CHECK` - runs betweenness a *second* time with a different random
  seed so we can measure how much the ranking depends on the sample. It roughly
  doubles the betweenness cost; set it to `False` if you are in a hurry, but the
  honesty argument in section 11 depends on it.
- `HARMONIC_SAMPLES = 300` - harmonic centrality is also estimated from sampled
  sources (each source costs one BFS). Set it to `None` for the exact computation,
  which is one BFS per node and takes tens of minutes.
- `TOP_N = 15` - length of the ranking tables and bar charts.

This notebook writes only into its own stage folder, `outputs/nb/04_centrality_analysis`.
It never touches `outputs/tables`, `outputs/figures` or `outputs/rail`, which hold the
results cited in the written report.

In [ ]:
_ensure("networkx", "pandas", "numpy", "matplotlib")

import csv
import json
import pickle
import random
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt

# ---------------- tunable constants (runtime lives here) ----------------
K_BETWEENNESS = 300          # sampled sources for approximate betweenness
BETWEENNESS_SEED = 42        # seed of the main sample
BETWEENNESS_SEED_B = 7       # seed of the second sample (noise check only)
RUN_STABILITY_CHECK = True   # False -> skip the second betweenness run
HARMONIC_SAMPLES = 300       # None -> exact harmonic centrality (very slow)
TOP_N = 15                   # rows per ranking table / bars per chart
CRITICAL_QUANTILE = 0.90     # the p90 rule we stress-test in section 11

# ---------------- stage folders ----------------
STAGE = OUT / "04_centrality_analysis"
TABLES = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
STAGE02 = OUT / "02_graph_construction"

print("stage folder :", STAGE)
print("reads stage  :", STAGE02)
print("networkx", nx.__version__, "| pandas", pd.__version__)

## 3. Hebrew labels in matplotlib

Station names in the Israeli GTFS feed are Hebrew. Matplotlib stores text in logical
order and does not apply the Unicode bidirectional algorithm, so a Hebrew label is
drawn left-to-right, i.e. visually reversed. The patch below wraps `Text.set_text` and
converts Hebrew strings to display order once, before anything is drawn. Latin text is
returned untouched, and the patch is idempotent so re-running the cell is harmless.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. Load the graph produced by notebook 02

This stage does not rebuild the graph from raw GTFS - that is notebook 02's job. Here
we read the node table (station attributes) and the directed edge table (one row per
`from_stop -> to_stop` segment with the number of trips that use it) and rebuild the
two `networkx` objects we need:

- `D`, a directed graph whose edge weight is the trip frequency of the segment. This is
  the object PageRank runs on, because direction matters for flow.
- `G`, the undirected projection whose edge weight is the sum of both directions. This
  is what degree, betweenness and harmonic centrality run on, because a passenger
  travelling between two adjacent stops does not care which direction the edge was
  recorded in.

The loader is deliberately tolerant about file names and column names (`weight` vs
`trip_frequency`, `source` vs `from_stop`, and so on) so that it keeps working if
notebook 02 renames something. If the stage-02 folder is missing entirely it raises an
actionable error instead of silently producing an empty graph.

In [ ]:
def _find_artifact(stage_dir, names, keyword):
    """Locate a stage artifact by exact name, then by keyword, under <stage>/tables or <stage>."""
    for name in names:
        for cand in (stage_dir / "tables" / name, stage_dir / name):
            if cand.exists():
                return cand
    hits = sorted(p for p in stage_dir.rglob("*.csv") if keyword in p.name.lower())
    return hits[0] if hits else None


def _pick_column(df, candidates):
    """Return the first column of df matching one of candidates (case-insensitive)."""
    lower = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    return None


if not STAGE02.exists():
    raise FileNotFoundError(
        f"{STAGE02} missing - run notebook 02_graph_construction first."
    )

edges_path = _find_artifact(
    STAGE02,
    ["edges.csv", "graph_edges.csv", "edges_directed.csv", "directed_edges.csv"],
    "edge",
)
nodes_path = _find_artifact(
    STAGE02, ["nodes.csv", "graph_nodes.csv", "stops_nodes.csv"], "node"
)
if edges_path is None:
    raise FileNotFoundError(
        f"No edge table found under {STAGE02} - run notebook 02_graph_construction first."
    )
print("edges from:", edges_path)
print("nodes from:", nodes_path)

edges_df = pd.read_csv(edges_path, dtype=str, encoding="utf-8-sig")
src_col = _pick_column(edges_df, ["from_stop", "from_stop_id", "source", "from", "u"])
dst_col = _pick_column(edges_df, ["to_stop", "to_stop_id", "target", "to", "v"])
wgt_col = _pick_column(
    edges_df, ["trip_frequency", "weight", "frequency", "trips", "n_trips", "count"]
)
if src_col is None or dst_col is None:
    raise ValueError(f"Cannot identify endpoint columns in {edges_path}: {list(edges_df.columns)}")
weights = (
    pd.to_numeric(edges_df[wgt_col], errors="coerce").fillna(1.0)
    if wgt_col is not None
    else pd.Series(1.0, index=edges_df.index)
)

# Directed trip graph: weight = number of trips using the segment.
D = nx.DiGraph()
for u, v, w in zip(edges_df[src_col], edges_df[dst_col], weights):
    if u == v or pd.isna(u) or pd.isna(v):
        continue
    if D.has_edge(u, v):
        D[u][v]["weight"] += float(w)
    else:
        D.add_edge(u, v, weight=float(w))

# Undirected projection: weight = sum of both directions.
G = nx.Graph()
for u, v, data in D.edges(data=True):
    if G.has_edge(u, v):
        G[u][v]["weight"] += data["weight"]
    else:
        G.add_edge(u, v, weight=data["weight"])

# Station attributes (name, coordinates, region, metro) if notebook 02 saved them.
ATTR = {}
VISIT_COL = None
if nodes_path is not None:
    nodes_df = pd.read_csv(nodes_path, dtype=str, encoding="utf-8-sig")
    id_col = _pick_column(nodes_df, ["stop_id", "node", "id"])
    name_col = _pick_column(nodes_df, ["stop_name", "name"])
    lat_col = _pick_column(nodes_df, ["lat", "stop_lat", "latitude"])
    lon_col = _pick_column(nodes_df, ["lon", "stop_lon", "longitude"])
    reg_col = _pick_column(nodes_df, ["region", "district"])
    met_col = _pick_column(nodes_df, ["metro", "metropolitan"])
    VISIT_COL = _pick_column(
        nodes_df,
        ["stop_use_count", "visits", "stop_visits", "visit_count", "n_visits"],
    )
    if id_col is not None:
        for row in nodes_df.itertuples(index=False):
            rec = dict(zip(nodes_df.columns, row))
            sid = rec[id_col]
            ATTR[sid] = {
                "stop_name": rec.get(name_col, "") if name_col else "",
                "lat": pd.to_numeric(rec.get(lat_col), errors="coerce") if lat_col else np.nan,
                "lon": pd.to_numeric(rec.get(lon_col), errors="coerce") if lon_col else np.nan,
                "region": rec.get(reg_col, "") if reg_col else "",
                "metro": rec.get(met_col, "") if met_col else "",
                "stop_use_count": pd.to_numeric(rec.get(VISIT_COL), errors="coerce")
                if VISIT_COL
                else np.nan,
            }
        # Keep stations that stage 02 listed even if they ended up with no segment.
        G.add_nodes_from(ATTR.keys())
        D.add_nodes_from(ATTR.keys())

print(f"directed graph  : {D.number_of_nodes():,} nodes, {D.number_of_edges():,} edges")
print(f"undirected graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"attributes for  : {len(ATTR):,} stations"
      + (f" (visit counts in column '{VISIT_COL}')" if VISIT_COL else " (no visit-count column)"))

## 5. Sanity check and the largest connected component

Two global centrality measures - betweenness and harmonic centrality - are only
meaningful between nodes that can actually reach each other, so both are computed on
the largest connected component (LCC) of `G`. Stations outside the LCC get a score of
0, which is the honest value: they are unreachable from the bulk of the network, so
they lie on no shortest path in it. We print the size of the LCC so the reader knows
how much of the network the global measures actually cover.

In [ ]:
components = sorted(nx.connected_components(G), key=len, reverse=True)
Gc = G.subgraph(components[0]).copy()

n_nodes = G.number_of_nodes()
print(f"connected components : {len(components):,}")
print(f"largest component    : {Gc.number_of_nodes():,} nodes "
      f"({Gc.number_of_nodes() / n_nodes:.1%} of the network), "
      f"{Gc.number_of_edges():,} edges")
print(f"isolated / off-LCC   : {n_nodes - Gc.number_of_nodes():,} stations")
print(f"average degree       : {2 * G.number_of_edges() / n_nodes:.2f}")
print(f"betweenness sampling : k = {min(K_BETWEENNESS, Gc.number_of_nodes()):,} sources "
      f"= {min(K_BETWEENNESS, Gc.number_of_nodes()) / Gc.number_of_nodes():.2%} of the LCC")

## 6. Degree and weighted degree

The cheapest measures, and the baseline everything else is compared against.

- `degree` - how many distinct neighbouring stations a stop is connected to. This
  counts *topological* branching: a stop served by many different corridors scores high.
- `degree_centrality` - the same number divided by `n - 1`, the networkx normalisation.
- `weighted_degree` - the sum of trip frequencies on all incident segments, i.e. the
  total daily service volume passing through the stop. This counts *traffic*, not
  branching: a single-corridor stop on a very frequent bus line scores high.
- `in_degree`, `out_degree`, `in_weight`, `out_weight` - the directed versions, taken
  from `D`. For a stop in the middle of a two-way corridor these are close to symmetric;
  a large asymmetry usually marks a terminus or a one-way loop.

All of these are dictionary lookups over the graph and cost essentially nothing.

In [ ]:
degree = dict(G.degree())
weighted_degree = dict(G.degree(weight="weight"))
in_degree = dict(D.in_degree())
out_degree = dict(D.out_degree())
in_weight = dict(D.in_degree(weight="weight"))
out_weight = dict(D.out_degree(weight="weight"))

print("degree      : max", max(degree.values()), "| mean", round(np.mean(list(degree.values())), 2))
print("weighted deg: max", int(max(weighted_degree.values())),
      "| mean", round(np.mean(list(weighted_degree.values())), 1))

## 7. Weighted PageRank on the directed graph

PageRank models a passenger doing a random walk along the network and, with probability
`1 - alpha = 0.15`, teleporting to a random station. A stop scores highly when many
*frequently served* segments lead into it from stops that are themselves important, so
PageRank is a flow-weighted notion of importance rather than a purely local count.

We run it on the **directed** graph `D` with `weight="weight"` (trip frequency), so that
one-way segments and asymmetric service are respected. Dangling nodes - terminal stops
with no outgoing segment - have their rank redistributed uniformly, which is the
standard treatment.

networkx's default `pagerank` uses SciPy. If SciPy is not installed we fall back to a
pure-Python power iteration (the same implementation used in the project's script
pipeline), so the notebook never fails on a bare environment. Either path costs a few
seconds.

In [ ]:
def weighted_pagerank(graph, weight="weight", alpha=0.85, max_iter=100, tol=1.0e-6):
    """Weighted PageRank via power iteration, without requiring SciPy."""
    nodes = list(graph.nodes)
    n = len(nodes)
    if n == 0:
        return {}
    rank = {node: 1.0 / n for node in nodes}
    out_w = {
        node: sum(data.get(weight, 1.0) for _, _, data in graph.out_edges(node, data=True))
        for node in nodes
    }
    for _ in range(max_iter):
        previous = rank
        dangling = sum(previous[node] for node in nodes if out_w[node] == 0)
        base = (1.0 - alpha) / n + alpha * dangling / n
        rank = {node: base for node in nodes}
        for source in nodes:
            total = out_w[source]
            if total == 0:
                continue
            src_rank = previous[source]
            for _, target, data in graph.out_edges(source, data=True):
                rank[target] += alpha * src_rank * data.get(weight, 1.0) / total
        if sum(abs(rank[node] - previous[node]) for node in nodes) < n * tol:
            return rank
    return rank


t0 = time.time()
try:
    pagerank = nx.pagerank(D, alpha=0.85, weight="weight")
    engine = "networkx (SciPy)"
except Exception as exc:  # SciPy missing or convergence failure
    print("networkx pagerank unavailable ->", exc, "| falling back to pure Python")
    pagerank = weighted_pagerank(D, weight="weight")
    engine = "pure Python power iteration"

print(f"PageRank via {engine} in {time.time() - t0:.1f}s; "
      f"sum = {sum(pagerank.values()):.4f} (should be ~1.0)")

## 8. Approximate betweenness centrality

Betweenness counts the fraction of shortest paths that pass through a station. It is the
measure that actually captures *bridging*: a stop with only two neighbours can still
have huge betweenness if it is the only way from one part of the country to another.
That is exactly the property a "critical station" analysis cares about.

It is also by far the most expensive measure. Exact Brandes betweenness is `O(n * m)`,
which for ~30,000 nodes and ~40,000 edges means a BFS from every node - hours of
compute. We therefore use networkx's sampled estimator: pick `k` random source nodes,
run the Brandes accumulation from those sources only, and rescale.

**Read this before using the numbers.** With `K_BETWEENNESS = 300` we sample about 1% of
the possible sources. The estimator is unbiased in expectation but has real variance:
the very top of the ranking (national bridges, which lie on shortest paths from almost
*any* source) is stable, while the middle and the tail move noticeably between samples.
Section 11 measures how much. Betweenness is computed on the LCC and unweighted -
"shortest" means fewest segments, not fastest or most frequent - which is the same
convention used everywhere else in this project.

In [ ]:
k_eff = min(K_BETWEENNESS, Gc.number_of_nodes())
t0 = time.time()
betweenness = nx.betweenness_centrality(
    Gc, k=k_eff, seed=BETWEENNESS_SEED, normalized=True, weight=None
)
print(f"approximate betweenness: k={k_eff}, seed={BETWEENNESS_SEED}, "
      f"{time.time() - t0:.1f}s")
nonzero = sum(1 for v in betweenness.values() if v > 0)
print(f"stations with a non-zero estimate: {nonzero:,} of {len(betweenness):,} "
      f"({nonzero / len(betweenness):.1%}) - the rest were simply never on a sampled path")

## 9. Harmonic centrality (sampled) on the largest component

Harmonic centrality sums `1 / d(u, v)` over all other stations `v`. Unlike closeness it
is well defined when some pairs are unreachable, and it answers a different question
from betweenness: not "how many routes pass through me" but "how close am I to
everything else", i.e. accessibility rather than bridging.

The exact computation is again one BFS per node. We therefore use the same sampling
trick as the project's script pipeline: draw `HARMONIC_SAMPLES` random sources, run a
BFS from each, accumulate `1 / d` into every reached target, and rescale by
`n_nodes / n_samples`. Because the graph is undirected, distance is symmetric, so
sampling sources is a valid unbiased estimate of the sum over targets. Set
`HARMONIC_SAMPLES = None` to compute it exactly (slow). Like betweenness, the sampled
version is noisy, but harmonic centrality is a smooth, non-heavy-tailed quantity, so it
is far better behaved under sampling than betweenness is.

In [ ]:
def approximate_harmonic_centrality(graph, samples, seed):
    """Estimate harmonic centrality from `samples` random BFS sources, rescaled to n."""
    if graph.number_of_nodes() == 0 or not samples:
        return {}
    rng = random.Random(seed)
    nodes = list(graph.nodes)
    sources = rng.sample(nodes, min(samples, len(nodes)))
    scores = defaultdict(float)
    for source in sources:
        lengths = nx.single_source_shortest_path_length(graph, source)
        for target, distance in lengths.items():
            if target == source or distance == 0:
                continue
            scores[target] += 1.0 / distance
    scale = graph.number_of_nodes() / len(sources)
    return {node: value * scale for node, value in scores.items()}


t0 = time.time()
if HARMONIC_SAMPLES is None:
    harmonic = nx.harmonic_centrality(Gc)
    harmonic_mode = "exact"
else:
    harmonic = approximate_harmonic_centrality(Gc, HARMONIC_SAMPLES, BETWEENNESS_SEED)
    harmonic_mode = f"sampled ({HARMONIC_SAMPLES} sources)"
print(f"harmonic centrality [{harmonic_mode}] in {time.time() - t0:.1f}s "
      f"for {len(harmonic):,} stations")

## 10. Assemble `stop_metrics.csv` and the top-N tables

Everything computed so far is keyed by `stop_id`; this cell joins it into one tidy table
with the station attributes (name, coordinates, region, metro) and writes it out. Stops
outside the LCC get `0.0` for betweenness and harmonic centrality, as explained in
section 5.

We also write one top-`TOP_N` table per measure. These are the tables the report quotes,
and keeping them as separate files makes it obvious which ranking a claim came from.

In [ ]:
default_attr = {"stop_name": "", "lat": np.nan, "lon": np.nan,
                "region": "", "metro": "", "stop_use_count": np.nan}

rows = []
for stop_id in G.nodes:
    a = ATTR.get(stop_id, default_attr)
    deg = degree.get(stop_id, 0)
    rows.append({
        "stop_id": stop_id,
        "stop_name": a.get("stop_name", ""),
        "region": a.get("region", ""),
        "metro": a.get("metro", ""),
        "lat": a.get("lat", np.nan),
        "lon": a.get("lon", np.nan),
        "stop_use_count": a.get("stop_use_count", np.nan),
        "degree": deg,
        "degree_centrality": deg / (n_nodes - 1) if n_nodes > 1 else 0.0,
        "weighted_degree": weighted_degree.get(stop_id, 0.0),
        "in_degree": in_degree.get(stop_id, 0),
        "out_degree": out_degree.get(stop_id, 0),
        "in_weight": in_weight.get(stop_id, 0.0),
        "out_weight": out_weight.get(stop_id, 0.0),
        "pagerank": pagerank.get(stop_id, 0.0),
        "approx_betweenness": betweenness.get(stop_id, 0.0),
        "approx_harmonic": harmonic.get(stop_id, 0.0),
        "in_largest_component": stop_id in Gc,
    })

metrics = pd.DataFrame(rows).sort_values("weighted_degree", ascending=False)
metrics["label"] = metrics["stop_name"].fillna("").astype(str).str.strip()
metrics["label"] = metrics["label"].where(metrics["label"] != "", metrics["stop_id"])

metrics.drop(columns=["label"]).to_csv(
    TABLES / "stop_metrics.csv", index=False, encoding="utf-8-sig"
)

RANKINGS = {
    "top_degree.csv": "degree",
    "top_weighted_degree.csv": "weighted_degree",
    "top_pagerank.csv": "pagerank",
    "top_approx_betweenness.csv": "approx_betweenness",
    "top_approx_harmonic.csv": "approx_harmonic",
}
for filename, column in RANKINGS.items():
    (metrics.drop(columns=["label"])
            .sort_values(column, ascending=False)
            .head(TOP_N)
            .to_csv(TABLES / filename, index=False, encoding="utf-8-sig"))

print(f"stop_metrics.csv written: {len(metrics):,} rows -> {TABLES}")
display(metrics.nlargest(10, "approx_betweenness")[
    ["stop_name", "stop_id", "region", "degree", "weighted_degree",
     "pagerank", "approx_betweenness"]
])

## 11. How noisy is the sampled betweenness? (the honesty section)

A ranking is only useful if it is reproducible. Here we re-run the *same* estimator with
a different random seed - nothing about the network changes, only which 300 sources were
drawn - and compare the two results three ways:

1. **Spearman correlation** between the two score vectors. High overall correlation is
   expected and does not prove much, because most stations score ~0 in both runs.
2. **Top-50 overlap.** How many of the top 50 stations of run A also appear in the top 50
   of run B. This is the number that matters for "which stations are critical".
3. **p90 threshold agreement.** A common downstream rule is "a station is critical if its
   betweenness is above the 90th percentile". We build that flag from each run and report
   the Jaccard overlap of the two flag sets. Whatever instability shows up here is
   inherited by every claim built on such a threshold.

This cell costs one extra betweenness run (~1-3 minutes). Set `RUN_STABILITY_CHECK = False`
to skip it.

In [ ]:
stability = None
if RUN_STABILITY_CHECK:
    t0 = time.time()
    betweenness_b = nx.betweenness_centrality(
        Gc, k=k_eff, seed=BETWEENNESS_SEED_B, normalized=True, weight=None
    )
    print(f"second betweenness run (seed={BETWEENNESS_SEED_B}) in {time.time() - t0:.1f}s")

    joint = pd.DataFrame({
        "seed_a": pd.Series(betweenness),
        "seed_b": pd.Series(betweenness_b),
    }).fillna(0.0)

    rho_all = joint["seed_a"].corr(joint["seed_b"], method="spearman")
    both_positive = joint[(joint["seed_a"] > 0) & (joint["seed_b"] > 0)]
    rho_pos = both_positive["seed_a"].corr(both_positive["seed_b"], method="spearman")

    top_a = set(joint.nlargest(50, "seed_a").index)
    top_b = set(joint.nlargest(50, "seed_b").index)
    overlap50 = len(top_a & top_b) / 50
    top_a10 = set(joint.nlargest(10, "seed_a").index)
    top_b10 = set(joint.nlargest(10, "seed_b").index)
    overlap10 = len(top_a10 & top_b10) / 10

    thr_a = joint["seed_a"].quantile(CRITICAL_QUANTILE)
    thr_b = joint["seed_b"].quantile(CRITICAL_QUANTILE)
    flag_a = joint["seed_a"] > thr_a
    flag_b = joint["seed_b"] > thr_b
    union = int((flag_a | flag_b).sum())
    jaccard = int((flag_a & flag_b).sum()) / union if union else float("nan")

    stability = pd.DataFrame([{
        "k_samples": k_eff,
        "lcc_nodes": Gc.number_of_nodes(),
        "sampling_fraction": k_eff / Gc.number_of_nodes(),
        "spearman_all_nodes": round(float(rho_all), 4),
        "spearman_both_positive": round(float(rho_pos), 4),
        "top10_overlap": overlap10,
        "top50_overlap": overlap50,
        "p90_flag_jaccard": round(float(jaccard), 4),
        "p90_flagged_seed_a": int(flag_a.sum()),
        "p90_flagged_seed_b": int(flag_b.sum()),
    }])
    stability.to_csv(TABLES / "betweenness_stability.csv", index=False, encoding="utf-8-sig")
    display(stability.T.rename(columns={0: "value"}))

    fig, ax = plt.subplots(figsize=(6.5, 6))
    ax.scatter(joint["seed_a"], joint["seed_b"], s=4, alpha=0.25, color="#dc2626")
    lim = max(joint["seed_a"].max(), joint["seed_b"].max()) * 1.05
    ax.plot([0, lim], [0, lim], color="#334155", lw=1, ls="--", label="perfect agreement")
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_xlabel(f"Approx. betweenness (seed {BETWEENNESS_SEED})")
    ax.set_ylabel(f"Approx. betweenness (seed {BETWEENNESS_SEED_B})")
    ax.set_title(f"Same estimator, different sample (k={k_eff})")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES / "betweenness_seed_stability.png", dpi=150)
    plt.show()
else:
    print("RUN_STABILITY_CHECK is False - skipping the betweenness noise measurement.")

## 12. Do the measures agree? Spearman correlation matrix

Spearman (rank) correlation is the right tool here because all of these distributions are
heavy-tailed: Pearson would be dominated by the handful of enormous stations. What we
want to know is whether the measures produce the same *ordering*.

Interpretation to keep in mind while reading the matrix:
- `degree` vs `weighted_degree` correlate strongly, but they are not the same thing -
  one is branching, the other is traffic.
- `pagerank` is largely a smoothed version of weighted in-degree, so a high correlation
  with `weighted_degree` is expected and is *not* an independent discovery.
- The interesting cell is `approx_betweenness` against the local measures. A moderate
  rather than high correlation is the substantive result: it means bridging stations are
  not simply the busiest stations. Remember that the betweenness column is the noisy
  sampled estimate, so its correlations are attenuated - the true correlation with
  exact betweenness would be somewhat higher.

In [ ]:
CORR_COLS = ["degree", "weighted_degree", "pagerank", "approx_betweenness", "approx_harmonic"]
corr = metrics[CORR_COLS].astype(float).corr(method="spearman").round(4)
corr.to_csv(TABLES / "centrality_correlation_spearman.csv", encoding="utf-8-sig")
display(corr)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(CORR_COLS)))
ax.set_xticklabels([c.replace("_", " ") for c in CORR_COLS], rotation=35, ha="right")
ax.set_yticks(range(len(CORR_COLS)))
ax.set_yticklabels([c.replace("_", " ") for c in CORR_COLS])
for i in range(len(CORR_COLS)):
    for j in range(len(CORR_COLS)):
        value = corr.values[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center",
                color="white" if abs(value) > 0.6 else "black", fontsize=10)
ax.set_title("Spearman correlation between centrality measures")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(FIGURES / "centrality_correlation_heatmap.png", dpi=150)
plt.show()

## 13. Top stations per measure

One horizontal bar chart per measure, showing the `TOP_N` highest-scoring stations with
their Hebrew names (rendered through the bidi patch from section 3; where a name is
missing we fall back to the raw `stop_id`). Bars are drawn at explicit y positions rather
than by category label, so two different stations that share a name do not collapse into
one bar - a real hazard here, since names like the same street in different cities repeat
across the feed.

Comparing the four charts is the point: if the same stations dominate every chart the
measures are redundant, and if the betweenness chart lists stations that appear nowhere
else then the network has structural bottlenecks that traffic volume alone would miss.

In [ ]:
def plot_top(df, column, title, color, filename, n=TOP_N):
    top = df.nlargest(n, column)[["label", column]].copy().sort_values(column)
    y = np.arange(len(top))
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(y, top[column].astype(float).values, color=color)
    ax.set_yticks(y)
    ax.set_yticklabels(top["label"].astype(str).values)
    ax.set_xlabel(column.replace("_", " ").title())
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=150)
    plt.show()


plot_top(metrics, "degree",
         f"Top {TOP_N} stations by degree (number of neighbouring stops)",
         "#2563eb", "top_degree.png")
plot_top(metrics, "weighted_degree",
         f"Top {TOP_N} stations by weighted degree (daily service volume)",
         "#0891b2", "top_weighted_degree.png")
plot_top(metrics, "pagerank",
         f"Top {TOP_N} stations by weighted PageRank",
         "#16a34a", "top_pagerank.png")
plot_top(metrics, "approx_betweenness",
         f"Top {TOP_N} stations by approximate betweenness (k={k_eff}, noisy)",
         "#dc2626", "top_approx_betweenness.png")
plot_top(metrics, "approx_harmonic",
         f"Top {TOP_N} stations by harmonic centrality ({harmonic_mode})",
         "#d97706", "top_approx_harmonic.png")

## 14. Where the measures disagree: scatter plots and a map

The correlation matrix gives one number per pair; these plots show the shape behind it.

- *Degree vs betweenness*: look for points low on the x axis and high on the y axis.
  Those are the low-connectivity, high-bridging stations - the ones a resilience analysis
  should worry about, and the ones a degree-based ranking would completely miss.
- *Degree vs PageRank*: expected to be much tighter, since PageRank is essentially a
  flow-weighted degree.
- The map plots every station faintly and highlights the 50 highest-betweenness stations,
  coloured by score. If those points line up along the Tel Aviv - Jerusalem - Haifa
  corridors rather than scattering, the estimate is picking up genuine national structure
  and not just sampling noise - a useful qualitative check on section 11.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(metrics["degree"], metrics["approx_betweenness"],
                s=4, alpha=0.3, color="#2563eb")
axes[0].set_xlabel("Degree")
axes[0].set_ylabel("Approx. betweenness")
axes[0].set_title("Degree vs betweenness (disagreement is the interesting part)")
axes[1].scatter(metrics["degree"], metrics["pagerank"],
                s=4, alpha=0.3, color="#16a34a")
axes[1].set_xlabel("Degree")
axes[1].set_ylabel("PageRank")
axes[1].set_yscale("log")
axes[1].set_title("Degree vs PageRank (log scale)")
fig.tight_layout()
fig.savefig(FIGURES / "centrality_scatter.png", dpi=150)
plt.show()

geo = metrics.dropna(subset=["lat", "lon"]).copy()
if len(geo):
    top_geo = geo.nlargest(50, "approx_betweenness")
    fig, ax = plt.subplots(figsize=(7.5, 10))
    ax.scatter(geo["lon"], geo["lat"], s=2, alpha=0.15, color="#94a3b8", label="all stations")
    sc = ax.scatter(top_geo["lon"], top_geo["lat"], s=45,
                    c=top_geo["approx_betweenness"], cmap="Reds",
                    edgecolors="black", linewidths=0.3, zorder=5)
    fig.colorbar(sc, ax=ax, label="Approx. betweenness", shrink=0.7)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Top 50 stations by approximate betweenness")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(FIGURES / "betweenness_map.png", dpi=150)
    plt.show()
else:
    print("No coordinates available in the stage-02 node table - skipping the map.")

## 15. Weighted degree vs stop visit count - a tautology, not a finding

It is tempting to report "weighted degree correlates ~0.99 with the number of times a
stop is visited by a trip, so busy stops are central". That correlation is real but it is
**true by construction and carries no information**.

Both quantities are built from the same rows of `stop_times.txt`. Every time a trip stops
at station `s` in the middle of its route it contributes exactly one incoming segment and
one outgoing segment, so

```
weighted_degree(s) = in_weight(s) + out_weight(s) ~= 2 * visits(s)
```

with deviations only at trip endpoints (a terminus contributes one side, not two) and at
consecutive duplicate stop entries, which the graph builder drops. So the near-unit
correlation is a restatement of the definition, roughly `y ~= 2x`, and cannot be used as
evidence for anything about the network.

The cell below verifies the identity `weighted_degree == in_weight + out_weight` exactly,
and reports the visit-count correlation *only* if stage 02 happened to save a visit-count
column (or if the 816 MB `stop_times.txt` is already on disk - we never download it just
for this check). Either way the number is printed as a sanity check, not as a result.

In [ ]:
# 1) Exact identity: undirected weighted degree = in_weight + out_weight.
identity_gap = (metrics["weighted_degree"]
                - (metrics["in_weight"] + metrics["out_weight"])).abs().max()
print(f"max |weighted_degree - (in_weight + out_weight)| = {identity_gap:.6f} "
      "(0 means the two are the same quantity)")

# 2) Visit counts, only if they are already available - no 816 MB download here.
visits = metrics["stop_use_count"]
if visits.notna().sum() == 0:
    stop_times_path = DATA / "stop_times.txt"
    if stop_times_path.exists():
        print("Counting stop visits from the local stop_times.txt (one streaming pass) ...")
        csv.field_size_limit(10_000_000)
        counts = Counter()
        with open(stop_times_path, encoding="utf-8-sig") as handle:
            reader = csv.reader(handle)
            header = next(reader)
            si = header.index("stop_id")
            for row in reader:
                counts[row[si]] += 1
        visits = metrics["stop_id"].map(counts)
    else:
        visits = pd.Series(np.nan, index=metrics.index)

if visits.notna().sum() > 0:
    paired = pd.DataFrame({"weighted_degree": metrics["weighted_degree"].astype(float),
                           "visits": visits.astype(float)}).dropna()
    rho = paired["weighted_degree"].corr(paired["visits"], method="spearman")
    ratio = (paired["weighted_degree"] / paired["visits"].replace(0, np.nan)).median()
    print(f"Spearman(weighted_degree, visits) = {rho:.4f} over {len(paired):,} stations")
    print(f"median weighted_degree / visits   = {ratio:.3f}  (~2 confirms the identity above)")
    print("Reminder: this is a definitional relationship, not an empirical finding.")
else:
    print("No visit-count column from stage 02 and no local stop_times.txt - "
          "skipping the numeric check. The identity in the markdown above still holds.")

## 16. Takeaways

Read these together with the tables in `outputs/nb/04_centrality_analysis/tables/`; the
exact numbers depend on the feed snapshot and on the sampling seeds.

1. **The measures are not interchangeable.** Degree, weighted degree and PageRank rank
   stations by how much service passes through them and largely agree with each other.
   Betweenness ranks them by how much of the network's connectivity depends on them, and
   agrees noticeably less. The stations that sit low-degree / high-betweenness in the
   scatter plot are the ones a resilience analysis should care about, and a traffic-based
   ranking would never surface them.

2. **The betweenness column is a 1% sample and must be treated as such.** With
   `K_BETWEENNESS = 300` sampled sources over a ~30,000-node component, the estimate is
   unbiased in expectation but individually noisy. Section 11 quantifies it by re-running
   with a second seed: the very top of the ranking is reasonably stable (the national
   corridor stations are on shortest paths from almost any source), but membership in the
   broader "top 50" and especially in a p90-threshold "critical" set moves between
   samples. **Any downstream rule of the form "betweenness above the 90th percentile means
   critical" inherits that instability**, and the cut-off should be treated as a fuzzy
   band rather than a hard line. Raising `K_BETWEENNESS` shrinks the variance roughly as
   `1/sqrt(k)` at linear cost; the exact computation would remove it entirely at a cost of
   hours.

3. **Weighted degree vs stop visit count is not a finding.** As shown in section 15,
   `weighted_degree = in_weight + out_weight ~= 2 * visits` by construction, both being
   derived from the same GTFS rows. The ~0.99 correlation restates a definition. It is
   worth reporting only as a data-integrity check that the graph was built correctly.

4. **Harmonic centrality adds a third, distinct view** - accessibility rather than volume
   or bridging - but it is smooth and geographically predictable: it mostly rewards being
   near the centre of the dense Gush Dan cluster. It is also estimated from sampled
   sources here, though it is far less sensitive to sampling than betweenness because it
   averages over many distances instead of concentrating on rare shortest paths.

5. **Coverage caveat.** Betweenness and harmonic centrality are defined only on the
   largest connected component; stations outside it are recorded as 0.0. That is the
   correct value for those measures, but it means the zero column mixes two very different
   situations - "never sampled" and "not in the main network" - and the
   `in_largest_component` flag in `stop_metrics.csv` is what tells them apart.